## **Ink-bleed detection — score distribution analysis**

Load the JSON produced by `run_ink_bleed_detection.py` and study the score distribution beyond what the run-summary captures by default. Use this to pick a stricter percentile (`p96`–`p99`) if the default `p75` flags too many images.

The detector writes:
- `summary.bleed_score_distribution` — min, p25, p50, p75, p95, max, mean.
- `summary.metric_normalization` — per-run min/max for the two sub-metrics (used during min-max rescale).
- `summary.effective_threshold` — the score cutoff the run actually applied (= the chosen percentile of `bleed_score`).
- `images[<rel_path>].bleed_score`, `.has_bleed`, `.metrics` — per-image record.

This notebook does **not** rerun detection. It only re-analyses an existing JSON, so you can pick a new percentile and see what would have been flagged without recomputing scores.

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

load_dotenv()
project_root = Path(os.environ.get("PROJECT_ROOT", "."))

# Edit this path to point at the run you want to analyse.
JSON_PATH = project_root / "data/processed/filtered_images/20260515_104416/ink_bleed_20260614_204740.json"

doc = json.loads(JSON_PATH.read_text(encoding="utf-8"))
summary = doc["summary"]
images  = doc["images"]

print(f"run: {summary['run']}")
print(f"images_dir: {summary['images_dir']}")
print(f"n_processed: {summary['n_processed']:,}  |  n_skipped: {summary['n_skipped']}")
print(f"chosen percentile: {summary.get('bleed_percentile', 'N/A')}")
print(f"effective_threshold: {summary.get('effective_threshold', 'N/A')}")
print(f"n_with_bleed (at effective threshold): {summary['n_with_bleed']:,}")

### Load per-image scores into a DataFrame

We keep just the columns we need for percentile analysis. `metrics.degenerate` is kept so we can flag single-tone images (their score is always 0 in this pipeline).

In [ ]:
rows = []
for rel_path, entry in images.items():
    m = entry["metrics"]
    rows.append({
        "path": rel_path,
        "bleed_score": entry["bleed_score"],
        "has_bleed": entry["has_bleed"],
        "bg_std_norm": m["bg_std_norm"],
        "intermediate_ratio": m["intermediate_ratio"],
        "degenerate": m.get("degenerate", False),
    })
df = pd.DataFrame(rows)
print(f"rows: {len(df):,}  |  degenerate: {df['degenerate'].sum():,}")
df.head()

### Extra percentiles: p96, p97, p98, p99

Compute the high-end percentiles on the **full score distribution** (matching what the detector does — degenerates included as score 0). For each, count how many images would be flagged if you used that percentile as the threshold.

In [ ]:
scores = df["bleed_score"].to_numpy()

wanted = [25, 50, 75, 95, 96, 97, 98, 99]
values = np.percentile(scores, wanted)

rows = []
for p, v in zip(wanted, values):
    # 'Flagged' means score strictly above the cutoff (matches the
    # detector's `> effective_threshold` rule).
    n_above = int((scores > v).sum())
    rows.append({
        "percentile": f"p{p}",
        "score_cutoff": round(float(v), 6),
        "n_flagged": n_above,
        "fraction_flagged": round(n_above / len(scores), 4),
    })
pct_df = pd.DataFrame(rows)
pct_df

### Compare with what the run actually flagged

Cross-check: the `n_flagged` row matching the run's chosen percentile should equal `summary['n_with_bleed']` (off-by-a-few due to floating-point ties on the boundary).

In [ ]:
chosen_p = summary.get("bleed_percentile", 75)
eff_thr  = summary.get("effective_threshold")

print(f"Run chose p{chosen_p} -> effective_threshold = {eff_thr}")
print(f"Recomputed p{chosen_p} from this notebook    = {np.percentile(scores, chosen_p):.6f}")
print(f"summary.n_with_bleed                          = {summary['n_with_bleed']:,}")
if eff_thr is not None:
    print(f"images with score > effective_threshold       = {int((scores > eff_thr).sum()):,}")

### Score distribution with percentile markers

Histogram of `bleed_score` with vertical lines at p75, p95, p96, p97, p98, p99. The denser the tail, the more flexibility you have to push the threshold up.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.hist(scores, bins=80, color="#8a98ad", edgecolor="white")

marker_pcts = [75, 95, 96, 97, 98, 99]
colors = ["#888", "#d77", "#c63", "#b53", "#a43", "#933"]
for p, c in zip(marker_pcts, colors):
    v = float(np.percentile(scores, p))
    ax.axvline(v, color=c, linestyle="--", linewidth=1.1, label=f"p{p} = {v:.3f}")

ax.set_xlabel("bleed_score")
ax.set_ylabel("image count")
ax.set_title("bleed_score distribution")
ax.legend(loc="upper right", framealpha=0.95)
ax.set_yscale("log")  # the tail is small; log-y makes p99 visible
plt.tight_layout()
plt.show()

### Top-20 worst images by `bleed_score`

Spot-check the highest-score images to confirm they actually look like bleed (and to catch any pathological case the heuristics get wrong).

In [ ]:
top = df.sort_values("bleed_score", ascending=False).head(20)
top[["path", "bleed_score", "bg_std_norm", "intermediate_ratio", "degenerate", "has_bleed"]]

### Sub-score correlation (sanity check)

If `bg_std_norm` and `intermediate_ratio` were perfectly correlated, the weighted combination wouldn't be doing much. A correlation below ~0.7 means the two signals capture genuinely different aspects of bleed and the weighted average is worth keeping.

In [ ]:
non_degen = df[~df["degenerate"]]
corr = non_degen[["bg_std_norm", "intermediate_ratio"]].corr().iloc[0, 1]
print(f"Pearson r(bg_std_norm, intermediate_ratio) = {corr:.3f}  on {len(non_degen):,} non-degenerate images")

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(non_degen["bg_std_norm"], non_degen["intermediate_ratio"],
           s=4, alpha=0.25, color="#3a4a6a")
ax.set_xlabel("bg_std_norm (raw)")
ax.set_ylabel("intermediate_ratio (raw)")
ax.set_title("raw sub-metrics — non-degenerate images")
plt.tight_layout()
plt.show()